# 6_ForecastSubmission
Code in this notebook uses the mosqlient package together with the provided mosqlimate API to submit forecasts for inclusion in the sprint.

In [1]:
import pandas as pd
import numpy as np
import torch 
import os
import json

import mosqlient as mos
import properscoring as ps
from darts import TimeSeries, concatenate
import matplotlib.pyplot as plt
import pickle
import helpers as hp

torch.set_float32_matmul_precision("medium")
pd.options.mode.copy_on_write = True
include_stringency_index = False
accel = "gpu"

quantiles = [0.01, 0.025, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.975, 0.99]
api_key = "hadrianang:cb1a6931-0b81-4fa7-bf78-380dc889eef9"

# 1. Reading in Forecast Results
Here, we read in a set of forecast results for submission.

In [2]:
tuning_name = "Tuning_30_07_2025" #Name of the tuning scenario to use
random_seed = 20

base_dir = os.getcwd()
model_input_dir = os.path.join(base_dir, "ModelInput_Round2")
tuning_dir = os.path.join(base_dir, "Dengue Tuning", tuning_name)
vis_output_dir = os.path.join(tuning_dir, "Results_Round2")
climate_tuning_dir = os.path.join(base_dir, "Climate Tuning", "Notebook_24_07_2025")
climate_model_name = "RandomForest"
climate_forecast_dir = os.path.join(climate_tuning_dir, climate_model_name, "Forecasts")

os.makedirs(tuning_dir, exist_ok = True)
os.makedirs(vis_output_dir, exist_ok = True)
weekly_cal = pd.read_csv(os.path.join(model_input_dir, "CalendarExt.csv"))
weekly_cal = weekly_cal.iloc[1:]

#Convert WeekStart to Date for joining later
weekly_cal["WeekStart"] = pd.to_datetime(weekly_cal["WeekStart"], format = "%d/%m/%Y")

climate_df = pd.read_csv(os.path.join(model_input_dir, "TimeVaryingCovs.csv"))
dengue_df = pd.read_csv(os.path.join(model_input_dir, "DengueCases.csv"))

#Remove uf = "ES" since there is no dengue data for Espirito Santo
climate_df = climate_df[climate_df["uf"] != "ES"]
dengue_df = dengue_df[dengue_df["uf"] != "ES"]

dengue_df = dengue_df[["epiweek", "Year", "Week", "uf", "uf_name", "Cases"]]

In [3]:
weekly_cal

,epiweek,WeekStart,WeekMid,WeekEnd,Year,Week,train_1,target_1,train_2,target_2,train_3,target_3,target_4
1,201001,2010-01-03,07/01/2010,09/01/2010,2010,1,True,False,True,False,True,False,False
2,201002,2010-01-10,14/01/2010,16/01/2010,2010,2,True,False,True,False,True,False,False
3,201003,2010-01-17,21/01/2010,23/01/2010,2010,3,True,False,True,False,True,False,False
4,201004,2010-01-24,28/01/2010,30/01/2010,2010,4,True,False,True,False,True,False,False
5,201005,2010-01-31,04/02/2010,06/02/2010,2010,5,True,False,True,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
883,202648,2026-11-29,03/12/2026,05/12/2026,2026,48,False,False,False,False,False,False,False
884,202649,2026-12-06,10/12/2026,12/12/2026,2026,49,False,False,False,False,False,False,False
885,202650,2026-12-13,17/12/2026,19/12/2026,2026,50,False,False,False,False,False,False,False
886,202651,2026-12-20,24/12/2026,26/12/2026,2026,51,False,False,False,False,False,False,False


In [4]:
uf_list = climate_df["uf"].unique()
uf_mapper = climate_df[["uf", "uf_name"]].copy().drop_duplicates()
uf_dict = dict(zip(uf_mapper["uf"], uf_mapper["uf_name"]))
uf_name_list = [uf_dict[curr_uf] for curr_uf in uf_list]

In [5]:
results_dirs = ["GlobalFit=False_TrainFromScratch=False_MultiClimate=True"]

#Read the predictions over the targets
targ_pred_dicts = [] #List of dictionaries containing predictions
for curr_dir in results_dirs:
    curr_filename = "Targ_" + curr_dir + ".pkl"
    loaded = hp.load_pickle(os.path.join(vis_output_dir, curr_filename))
    targ_pred_dicts.append(loaded)

target_keys = ["targ1_preds", "targ2_preds", "targ3_preds", "targ4_preds"]

In [6]:
targ_pred_dicts[0].keys()

dict_keys(['targ1_preds', 'targ2_preds', 'targ3_preds', 'targ4_preds', 'targ1_preds_log_s', 'targ2_preds_log_s', 'targ3_preds_log_s', 'targ4_preds_log_s', 'den_eval_scaler'])

# 2. Model Submission

In [7]:
name = "Imperial-TFT Model"
description = "Temporal fusion transformer model for sprint 2025"
repository = "https://github.com/hadrianang/imperial-mosqlimate-sprint2025"
implementation_language = "Python"
disease = "dengue"
temporal = True
spatial = False
categorical = False
adm_level = 1
time_resolution = "week"
sprint = True

In [8]:
# mos.upload_model(api_key = api_key,
#                  name = name,
#                  description = description,
#                  repository = repository,
#                  implementation_language = implementation_language,
#                  disease = disease,
#                  temporal = temporal,
#                  spatial = spatial,
#                  categorical = categorical,
#                  adm_level = adm_level,
#                  time_resolution = time_resolution,
#                  sprint = sprint
#                 )

# 3. Forecast Submission

In [9]:
quantile_output = {
    "lower_95": 0.025,
    "lower_90": 0.05,
    "lower_80": 0.10,
    "lower_50": 0.25,
    "pred": 0.5,
    "upper_50": 0.75,
    "upper_80": 0.90,
    "upper_90": 0.95,
    "upper_95": 0.975
}
quantile_keys = list(quantile_output.keys())
#commit_hash = "a6e73530f7b447425bc027231058798a231bdc15"
commit_hash = "b08dfe47d50ff66b2af7f9c7257450d29f90c00f"
model_id = 136
predict_date = pd.to_datetime("23-09-2025", dayfirst = True)

In [10]:
curr_pred_dict = targ_pred_dicts[0]
# curr_preds_list = curr_pred_dict["targ3_preds"]
# curr_set = "target_3"

In [11]:
def format_sub_dfs(curr_preds_list, curr_set):
    to_ret = []
    for curr_ind in range(len(curr_preds_list)):
        curr_preds = curr_preds_list[curr_ind]
        #curr_uf = uf_list[curr_ind]
        curr_result_builer = [] 
        #curr_quantile_key = quantile_keys[0]
        for curr_quantile_key in quantile_keys:
            curr_preds_q = curr_preds.quantile(quantile_output[curr_quantile_key])
            curr_preds_q_df = curr_preds_q.to_dataframe()
            #curr_preds_q_df = curr_preds_q.map(np.ceil).to_dataframe() #Get the ceiling to prevent very small negative numbers and then convert to DataFrame
            curr_preds_q_df.columns = [curr_quantile_key]
            
            #To prevent negative values, we could also convert negative values to 0
            curr_preds_q_df[curr_quantile_key] = curr_preds_q_df.apply(lambda row: 0 if (row[curr_quantile_key] < 0) else row[curr_quantile_key], axis = 1)
            
            # if curr_quantile_key == "lower_95":
            #     neg_rows = curr_preds_q_df[curr_preds_q_df["lower_95"] < 0]
            #     if len(neg_rows) > 0:
            #         print("FOUND NEG ROWS: " + str(curr_ind))
            curr_result_builer.append(curr_preds_q_df)
        curr_result_df = pd.concat(curr_result_builer, axis = 1)
        curr_result_df.reset_index()
        curr_result_df = curr_result_df.merge(weekly_cal[["WeekStart", curr_set]], how = "left", on = "WeekStart")
        curr_result_df.rename(columns={'WeekStart': 'date'}, inplace=True)
        curr_result_df = curr_result_df[curr_result_df[curr_set]]
        curr_result_df = curr_result_df.drop(curr_set, axis = 1)
        to_ret.append(curr_result_df)
    return to_ret

In [12]:
targ1_res_dfs = format_sub_dfs(curr_pred_dict["targ1_preds"], curr_set = "target_1")
targ2_res_dfs = format_sub_dfs(curr_pred_dict["targ2_preds"], curr_set = "target_2")
targ3_res_dfs = format_sub_dfs(curr_pred_dict["targ3_preds"], curr_set = "target_3")
targ4_res_dfs = format_sub_dfs(curr_pred_dict["targ4_preds"], curr_set = "target_4")

In [13]:
targ4_res_dfs[0].shape

(53, 10)

In [15]:
# submit1_res = []
# for curr_uf, curr_submit_df in zip(uf_list, targ1_res_dfs):
#     description = "TFT forecasts for " + curr_uf
#     temp = mos.upload_prediction(api_key = api_key, 
#                                  model_id = model_id,
#                                  description = description,
#                                  commit = commit_hash,
#                                  predict_date = predict_date,
#                                  prediction = curr_submit_df,
#                                  adm_1 = curr_uf
#                                 )
#     submit1_res.append(temp)

In [16]:
#submit1_res

In [17]:
# submit2_res = []
# for curr_uf, curr_submit_df in zip(uf_list, targ2_res_dfs):
#     description = "TFT forecasts for " + curr_uf
#     temp = mos.upload_prediction(api_key = api_key, 
#                                  model_id = model_id,
#                                  description = description,
#                                  commit = commit_hash,
#                                  predict_date = predict_date,
#                                  prediction = curr_submit_df,
#                                  adm_1 = curr_uf
#                                 )
#     submit2_res.append(temp)

In [18]:
# submit3_res = []
# for curr_uf, curr_submit_df in zip(uf_list, targ3_res_dfs):
#     description = "TFT forecasts for " + curr_uf
#     temp = mos.upload_prediction(api_key = api_key, 
#                                  model_id = model_id,
#                                  description = description,
#                                  commit = commit_hash,
#                                  predict_date = predict_date,
#                                  prediction = curr_submit_df,
#                                  adm_1 = curr_uf
#                                 )
#     submit3_res.append(temp)

In [20]:
submit4_res = []
for curr_uf, curr_submit_df in zip(uf_list, targ4_res_dfs):
    description = "TFT forecasts for " + curr_uf
    temp = mos.upload_prediction(api_key = api_key, 
                                 model_id = model_id,
                                 description = description,
                                 commit = commit_hash,
                                 predict_date = predict_date,
                                 prediction = curr_submit_df,
                                 adm_1 = curr_uf
                                )
    submit4_res.append(temp)

In [21]:
submit4_res

[Prediction <4557>,
 Prediction <4558>,
 Prediction <4559>,
 Prediction <4560>,
 Prediction <4561>,
 Prediction <4562>,
 Prediction <4563>,
 Prediction <4564>,
 Prediction <4565>,
 Prediction <4566>,
 Prediction <4567>,
 Prediction <4568>,
 Prediction <4569>,
 Prediction <4570>,
 Prediction <4571>,
 Prediction <4572>,
 Prediction <4573>,
 Prediction <4574>,
 Prediction <4575>,
 Prediction <4576>,
 Prediction <4577>,
 Prediction <4578>,
 Prediction <4579>,
 Prediction <4580>,
 Prediction <4581>,
 Prediction <4582>]

In [22]:
targ4_res_dfs[0]

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95
15,2025-10-05,3.240131,4.710113,6.769047,11.107535,19.843079,35.620680,54.967266,82.012323,95.464024
16,2025-10-12,3.980980,6.653055,7.833851,12.472774,20.810665,41.240758,62.235401,90.456731,107.303293
17,2025-10-19,2.746002,3.508078,6.764193,11.218907,20.764334,33.923186,49.828102,67.171314,102.646642
18,2025-10-26,3.963485,6.072749,7.553120,12.248514,23.133389,41.510687,66.845523,80.011985,103.771014
19,2025-11-02,3.425937,5.202918,7.090534,14.375865,23.875911,43.430587,69.979073,105.134684,130.566764
20,2025-11-09,4.099778,7.639033,9.918100,16.376119,26.047685,45.587353,66.054712,98.662761,144.412083
21,2025-11-16,3.800282,5.808588,8.742190,13.919106,26.703398,42.988967,80.022112,103.679706,128.628878
22,2025-11-23,3.088974,5.243979,10.730079,17.374309,26.733990,43.504178,72.307212,111.863924,145.658687
23,2025-11-30,5.675317,7.419442,9.533848,16.698483,30.891167,54.144995,81.076568,115.765922,147.806642
24,2025-12-07,4.855435,5.452881,7.763785,16.876764,28.211052,52.596479,80.255106,111.007556,133.418401
